# statute2md · Bundesrecht für Obsidian

1. Führe die GUI-Zelle aus. Colab verwendet die Arbeitskopie unter `/content/statute2md` und klont sie bei Bedarf. Hinterlege in den Colab-Secrets `gittoken` (GitHub-Token) und `github_username` (dein GitHub-Benutzername, z. B. `rps007`) und aktiviere für beide den Notebook-Zugriff. Der Code verwendet das Repository `<github_username>/statute2md` auf `main` für Clone, Pull, Commit und Push. `gitemail` setzt optional die Commit-E-Mail.
2. **Add Sources** lädt zum eingegebenen Kürzel das Originalarchiv und legt die XML-Datei in `xml-sources/` ab. Falls Colab die Quelle nicht erreicht, lade das Original-ZIP im Browser herunter. Lade es über die Colab-Dateileiste nach `/content/statute2md/xml-sources` hoch und trage den Dateinamen bei **ZIP-Pfad** ein. Ein vollständiger Pfad ist ebenfalls möglich.
3. Wähle unter **Dateioperationen auf Colab** eine oder mehrere XML-Dateien und klicke **Transform**. Nach der Prüfung und Speicherung zeigt der Balken 100 % und eine Abschlussmeldung. Die Exporte liegen in `/content/statute2md/md_files`. Neue Gesetze werden mit den vorhandenen Exporten gemeinsam verlinkt.
4. **Delete** löscht die ausgewählten XML- und Markdown-Dateien. **Download** lädt eine ausgewählte Datei einzeln oder mehrere zusammen als ZIP herunter. **Obsidian-ZIP** lädt den Gesamtbestand. Entpacke dessen Ordner `Bundesrecht/` direkt in die Vault-Wurzel.
5. Unter **Anbindung an Github** aktualisiert **Clone** ausschließlich die bestehende Arbeitskopie unter `/content/statute2md` auf `main`. **Commit** erfasst Dateiänderungen in `xml-sources/` und `md_files/`. **Push** überträgt die Commits auf `main`.


## GUI starten
Führe die folgende Zelle aus. Sie öffnet die Dateiverwaltung und die Git-Aktionen.


In [ ]:
import base64
from pathlib import Path
import importlib
import os
import re
import shutil
from tempfile import mkdtemp
import subprocess
import sys


def update_code_checkout(root, remote, env):
    def run(command, *, check=True):
        try:
            result = subprocess.run(command, env=env, check=False, capture_output=True,
                                    text=True, errors="replace", timeout=45)
        except subprocess.TimeoutExpired:
            raise RuntimeError("Git hat nach 45 Sekunden abgebrochen. Prüfe die Verbindung zu GitHub.") from None
        if check and result.returncode:
            detail = result.stderr.strip() or result.stdout.strip() or "Keine weitere Git-Fehlermeldung."
            # Zugangsdaten dürfen auch in einer Git-Fehlermeldung nicht erscheinen.
            for name, value in env.items():
                if name.startswith("GIT_CONFIG_VALUE_") and value.startswith("Authorization: Basic "):
                    encoded = value.removeprefix("Authorization: Basic ")
                    detail = detail.replace(encoded, "[Token entfernt]")
                    raw = base64.b64decode(encoded).decode().partition(":")[2]
                    if raw:
                        detail = detail.replace(raw, "[Token entfernt]")
            raise RuntimeError(f"Git fehlgeschlagen (Exit-Code {result.returncode}): {detail}") from None
        return result

    if not root.exists():
        print("Git · clone · Zeitlimit 45 Sekunden.", flush=True)
        run(["git", "clone", "--branch", "main", "--", remote, str(root)])
        return
    git = ["git", "-C", str(root)]
    print("Git · fetch · Zeitlimit 45 Sekunden.", flush=True)
    run([*git, "fetch", "--no-tags", remote, "main"])
    common = run([*git, "merge-base", "HEAD", "FETCH_HEAD"], check=False)
    if common.returncode == 0:
        run([*git, "merge", "--ff-only", "FETCH_HEAD"])
        return
    if common.returncode != 1:
        raise RuntimeError("Der lokale Git-Verlauf konnte nicht geprüft werden. Die Arbeitskopie bleibt erhalten.")

    recovery = Path(mkdtemp(prefix="statute2md-recovery-", dir=root.parent))
    stage, backup = recovery / "checkout", recovery / "previous"
    print(f"Git-Verlauf wurde ersetzt. Wiederherstellungsordner: {recovery}", flush=True)
    run(["git", "clone", "--branch", "main", "--", remote, str(stage)])
    for name in ("xml-sources", "md_files"):
        source = root / name
        if source.exists() or source.is_symlink():
            if source.is_symlink() or not source.is_dir():
                raise RuntimeError(f"{name} ist kein regulärer Datenordner. Die ursprüngliche Arbeitskopie bleibt erhalten.")
            shutil.copytree(source, stage / name, dirs_exist_ok=True, symlinks=True)
    try:
        relative_cwd = Path.cwd().resolve().relative_to(root.resolve())
    except ValueError:
        relative_cwd = None
    root.rename(backup)
    try:
        stage.rename(root)
    except OSError:
        backup.rename(root)
        raise
    if relative_cwd is not None:
        destination = root / relative_cwd
        os.chdir(destination if destination.is_dir() else root)
    print(f"Arbeitskopie auf main erneuert; xml-sources und md_files übernommen. Vollständige alte Arbeitskopie: {backup}", flush=True)


def start_statute2md():
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    def secret(name):
        if userdata is None:
            return None
        try:
            return userdata.get(name)
        except (userdata.NotebookAccessError, userdata.SecretNotFoundError, userdata.TimeoutException):
            return None

    root = Path("/content/statute2md") if userdata is not None else Path.cwd().resolve()
    previous = globals().get("statute_gui")
    if previous is not None:
        if previous.running:
            raise RuntimeError("Eine Aktion läuft noch. Wenn die alte GUI hängt, starte die Colab-Laufzeit neu.")
        if userdata is None:
            root = previous.repo.root

    if userdata is not None:
        print("Lese Colab-Secrets für Git …", flush=True)
    token = secret("gittoken")
    email = secret("gitemail")
    username = secret("github_username")
    if userdata is not None and (not token or not username):
        raise RuntimeError("Hinterlege gittoken und github_username in den Colab-Secrets und gib dem Notebook Zugriff.")
    username = (username or "rps007").strip()
    if not re.fullmatch(r"[A-Za-z0-9](?:[A-Za-z0-9-]{0,37}[A-Za-z0-9])?", username):
        raise RuntimeError("github_username muss ein gültiger GitHub-Benutzername sein.")
    remote = f"https://github.com/{username}/statute2md.git"
    gui_module = root / "repository_gui.py"
    needs_update = not gui_module.is_file() or "NOTEBOOK_GUI_API = 15" not in gui_module.read_text(encoding="utf-8")
    if userdata is not None and root.exists():
        git = ["git", "-C", str(root)]
        origin = subprocess.check_output([*git, "remote", "get-url", "origin"], text=True, timeout=10).strip()
        branch = subprocess.check_output([*git, "branch", "--show-current"], text=True, timeout=10).strip()
        if origin.removesuffix(".git") != remote.removesuffix(".git") or branch != "main":
            raise RuntimeError(f"Die Colab-Arbeitskopie muss auf main stehen und auf {username}/statute2md verweisen. Sichere vorhandene Daten und verwende eine passende Arbeitskopie.")
    if needs_update and userdata is not None:
        credential = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GCM_INTERACTIVE="never", GIT_CONFIG_COUNT="1",
                   GIT_CONFIG_KEY_0=f"http.{remote}.extraHeader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {credential}")
        update_code_checkout(root, remote, env)

    if not (root / "repository_gui.py").is_file():
        raise RuntimeError("Starte Jupyter aus dem geklonten statute2md-Ordner.")
    log_outputs = previous.log.outputs if previous is not None else ()
    if previous is not None:
        previous.close()
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    from repository_setup import ensure_widgets
    ensure_widgets()
    import statute_source
    import statute_model
    import statute_links
    import statute_export
    import repository_files
    import repository_git
    import repository_gui
    if needs_update or getattr(repository_gui, "NOTEBOOK_GUI_API", None) != 15:
        importlib.reload(statute_source)
        importlib.reload(statute_model)
        importlib.reload(statute_links)
        importlib.reload(statute_export)
        importlib.reload(repository_files)
        importlib.reload(repository_git)
        importlib.reload(repository_gui)
    return repository_gui.start(root, token=token, email=email, username=username, log_outputs=log_outputs)


statute_gui = start_statute2md()
